In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

import pandas as pd
import numpy as np

from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [ ]:
results_dir = Path("/Users/sylvi/topo_data/dna_damage_cache/analysis_results")
results_csv_file = results_dir / "defect_grain_statistics.csv"
assert results_csv_file.exists(), f"Results CSV file does not exist: {results_csv_file}"
df = pd.read_csv(results_csv_file)
print(df.columns)
original_df = df.copy()

In [ ]:
cols_to_pca = [
    "num_height_defects",
    "num_curvature_defects",
    "coinciding_defects",
    # "num_beak_defects",
    "curvature_min",
    "curvature_std",
    "curvature_var",
    "curvature_total",
    "curvature_median",
    "curvature_iqr",
    "curvature_90th",
]
df = df[cols_to_pca]

pipe = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median")),  # impute missing values with the median of each column
        ("scale", StandardScaler()),  # scale the data to have mean 0 and variance 1
        ("pca", PCA(n_components=2)),  # reduce the data to 2 principal components
    ]
)

# fit the pipeline to the data, scores are the PCA transformed values
scores = pipe.fit_transform(df)  # run the pipeline on the data
pca = pipe.named_steps["pca"]  # get the PCA step from the pipeline

pc_df = pd.DataFrame(scores, columns=["PC1", "PC2"], index=df.index)

# add columns from the original dataframe to the PCA dataframe
pc_df["grain_id"] = original_df["grain_id"]
pc_df["folder"] = original_df["folder"]

# get the loadings - ie the contribution of each original feature to the PCs.
loadings = pd.DataFrame(pca.components_.T, index=df.columns, columns=["PC1", "PC2"])

print(f"explained variance ratio: {pca.explained_variance_ratio_}")
print(f"loadings: {loadings.sort_values(by='PC1', ascending=False).head(10)}")

for folder in pc_df["folder"].unique():
    folder_df = pc_df[pc_df["folder"] == folder]
    plt.scatter(folder_df["PC1"], folder_df["PC2"], label=folder)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
plt.legend()
plt.show()